In [1]:
!nvidia-smi

Thu Apr  9 16:11:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/GwenTsang/EMOTYC

Cloning into 'EMOTYC'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 174 (delta 51), reused 150 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (174/174), 4.80 MiB | 10.00 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [3]:
%cd /content/EMOTYC

/content/EMOTYC


In [6]:
# Install dependencies needed by error_analysis.py
!pip install -q shap mlxtend transformers accelerate openpyxl

In [14]:
%%writefile /content/patch_script.py
"""Patch error_analysis.py for Colab compatibility."""
import re

path = "/content/EMOTYC/experimentations/error_analysis.py"
with open(path, "r") as f:
    code = f.read()

# 1) Fix pct_uppercase lambda (NaN TEXT)
code = code.replace(
    "lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1)",
    "lambda t: sum(1 for c in str(t) if c.isupper()) / max(len(str(t)), 1)"
)

# 2) Fix TEXT fillna
code = code.replace(
    'df_all["TEXT"] = df_all["TEXT"].astype(str)',
    'df_all["TEXT"] = df_all["TEXT"].fillna("").astype(str)'
)

# 3) Remove HF_HUB_OFFLINE (Colab needs to download the model)
code = re.sub(r'.*HF_HUB_OFFLINE.*\n', '', code)

# 4) Remove standalone "import torch" at top level
code = re.sub(r'^# Pre-import torch.*\n', '', code, flags=re.MULTILINE)
code = re.sub(r'^import torch\n', '', code, flags=re.MULTILINE)

# 5) Fix fillna on Categorical columns in build_analysis_features
code = code.replace(
    '''        series = df[col].copy()
        series = series.fillna("MISSING")''',
    '''        series = df[col].astype(str).replace("nan", "MISSING").fillna("MISSING")'''
)

with open(path, "w") as f:
    f.write(code)

print("✓ All patches applied")

Writing /content/patch_script.py


In [12]:
import subprocess
r = subprocess.run(
    ["python", "/content/EMOTYC/experimentations/error_analysis.py", "--device", "cuda"],
    capture_output=True, text=True
)
# Write full output to file for reference
with open("/content/pipeline_log.txt", "w") as f:
    f.write("=== STDOUT ===\n")
    f.write(r.stdout)
    f.write("\n=== STDERR ===\n")
    f.write(r.stderr)

# Print only the last 40 lines of stdout + any stderr
stdout_lines = r.stdout.strip().split("\n")
print(f"[exit code: {r.returncode}]")
print(f"[stdout: {len(stdout_lines)} lines total, showing last 40]")
print("---")
for line in stdout_lines[-40:]:
    print(line)
if r.stderr.strip():
    stderr_lines = r.stderr.strip().split("\n")
    # Filter out common warnings
    err_lines = [l for l in stderr_lines if "FutureWarning" not in l and "UserWarning" not in l]
    if err_lines:
        print(f"\n=== STDERR (filtered, {len(err_lines)} lines) ===")
        for line in err_lines[-30:]:
            print(line)

[exit code: 1]
[stdout: 35 lines total, showing last 40]
---
══════════════════════════════════════════════════════════════════════
  EMOTYC Error Analysis Pipeline
  Config: context=no, thresholds=optimized
══════════════════════════════════════════════════════════════════════

▸ 1. Chargement et nettoyage des données…
  ✓ Homophobie: 103 lignes, 59 colonnes
  ✓ Obésité: 373 lignes, 58 colonnes
  ✓ Racisme: 201 lignes, 58 colonnes
  ✓ Religion: 104 lignes, 72 colonnes

  Total : 781 lignes
  Après nettoyage : 781/781 lignes avec ROLE valide

▸ 2. Prédictions EMOTYC…
  ✓ Modèle EMOTYC chargé sur cuda

  Inférence sur 781 textes (batch_size=16)…
  ✓ Inférence terminée — shape: (781, 19)

▸ 3. Calcul des métriques d'erreur…

  ═══ Résumé des erreurs (11 émotions, K=11) ═══
  Hamming moyen        : 0.0541
  Hamming médian       : 0.0000
  Jaccard error moyen  : 0.4631
  Exact match rate     : 0.5160
  Weighted Hamming moy : 0.0098
  Distribution n_errors:
    0 erreurs:  51.6%
    1 erreu

In [15]:
# Apply patches then re-run
!python /content/patch_script.py

import subprocess
r = subprocess.run(
    ["python", "/content/EMOTYC/experimentations/error_analysis.py", "--device", "cuda"],
    capture_output=True, text=True
)
with open("/content/pipeline_log.txt", "w") as f:
    f.write("=== STDOUT ===\n")
    f.write(r.stdout)
    f.write("\n=== STDERR ===\n")
    f.write(r.stderr)

stdout_lines = r.stdout.strip().split("\n")
print(f"[exit code: {r.returncode}]")
print(f"[stdout: {len(stdout_lines)} lines]")
print("--- LAST 40 LINES ---")
for line in stdout_lines[-40:]:
    print(line)
if r.returncode != 0:
    stderr_lines = r.stderr.strip().split("\n")
    err_lines = [l for l in stderr_lines if "Warning" not in l and "warn" not in l.lower()]
    print(f"\n=== STDERR ({len(err_lines)} error lines) ===")
    for line in err_lines[-20:]:
        print(line)

✓ All patches applied
[exit code: 1]
[stdout: 44 lines]
--- LAST 40 LINES ---

▸ 1. Chargement et nettoyage des données…
  ✓ Homophobie: 103 lignes, 59 colonnes
  ✓ Obésité: 373 lignes, 58 colonnes
  ✓ Racisme: 201 lignes, 58 colonnes
  ✓ Religion: 104 lignes, 72 colonnes

  Total : 781 lignes
  Après nettoyage : 781/781 lignes avec ROLE valide

▸ 2. Prédictions EMOTYC…
  ✓ Modèle EMOTYC chargé sur cuda

  Inférence sur 781 textes (batch_size=16)…
  ✓ Inférence terminée — shape: (781, 19)

▸ 3. Calcul des métriques d'erreur…

  ═══ Résumé des erreurs (11 émotions, K=11) ═══
  Hamming moyen        : 0.0541
  Hamming médian       : 0.0000
  Jaccard error moyen  : 0.4631
  Exact match rate     : 0.5160
  Weighted Hamming moy : 0.0098
  Distribution n_errors:
    0 erreurs:  51.6%
    1 erreurs:  37.5%
    2 erreurs:  10.6%
    3 erreurs:   0.3%

▸ 4. Construction des features d'analyse…
  Matrice de features : 781 lignes × 56 features
  ✓ DataFrame exporté : /content/EMOTYC/experimentatio

In [16]:
# Check final status from the log
with open("/content/pipeline_log.txt") as f:
    lines = f.readlines()
print(f"Log: {len(lines)} lines")
print("--- LAST 35 ---")
for l in lines[-35:]:
    print(l.rstrip())

Log: 494 lines
--- LAST 35 ---
/content/EMOTYC/experimentations/error_analysis.py:495: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"err_{emo}"] = np.where(
/content/EMOTYC/experimentations/error_analysis.py:495: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"err_{emo}"] = np.where(
/content/EMOTYC/experimentations/error_analysis.py:501: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u

In [ ]:
import os
out_dir = "/content/EMOTYC/experimentations/error_analysis_results"
if os.path.isdir(out_dir):
    for root, dirs, files in os.walk(out_dir):
        level = root.replace(out_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for f in files:
            size = os.path.getsize(os.path.join(root, f))
            print(f'{subindent}{f} ({size:,} bytes)')
    total = sum(os.path.getsize(os.path.join(r,f)) for r,dirs,files in os.walk(out_dir) for f in files)
    print(f"\nTotal: {total:,} bytes")
else:
    print("Directory not found!")

# Also show last 10 lines of log
with open("/content/pipeline_log.txt") as f:
    lines = f.readlines()
print(f"\n--- Log tail (last 15 of {len(lines)} lines) ---")
for l in lines[-15:]:
    print(l.rstrip())